# Train on a free Colab GPU
Runtime → Change runtime type → **T4 GPU**. No Google Drive needed: the first cell clones the repo. Download `results.zip` at the end, because Colab files are deleted when the session ends.

In [ ]:
%cd /content
!git clone https://github.com/Archu9999/clinical-slm-extraction.git
%cd /content/clinical-slm-extraction

In [ ]:
!pip install -q -U "transformers==5.18.0" "peft==0.21.2" "trl==1.14.1" "accelerate==1.15.0" "datasets==5.0.1" anthropic requests pytest
!pip uninstall -y -q torchao  # Colab ships an old torchao that PEFT rejects
!pytest -q tests

In [ ]:
# data is already in the repo; uncomment to regenerate
# !python src/make_data.py --out data

## 1. Baseline: base model with no fine-tuning (the "before" number)

In [ ]:
!python src/predict_local.py --base Qwen/Qwen2.5-1.5B-Instruct --batch-size 16 --gpu-usd-per-hour 0.35 --out results/preds_base.jsonl

## 2. Fine-tune with LoRA (~13 min on a T4, 125 steps; it should print dtype=torch.float16)

In [ ]:
!python src/train_lora.py --base Qwen/Qwen2.5-1.5B-Instruct --out checkpoints/qwen-1.5b-lora --epochs 1

## 3. Fine-tuned model (the "after" number)

In [ ]:
!python src/predict_local.py --adapter checkpoints/qwen-1.5b-lora --batch-size 16 --gpu-usd-per-hour 0.35 --out results/preds_lora.jsonl

## 4. Compare
Run `baseline_claude.py` on your laptop (it needs your API key), copy the preds file into `results/`, then add it here.

In [ ]:
!python src/evaluate.py base=results/preds_base.jsonl lora=results/preds_lora.jsonl --errors 3

## 5. (Optional) Serve with vLLM for real throughput numbers

In [ ]:
!pip install -q vllm
!nohup bash serve_vllm.sh checkpoints/qwen-1.5b-lora > vllm.log 2>&1 &
# wait ~2 min for the server to load, then:
!python src/predict_local.py --backend vllm --gpu-usd-per-hour 0.35 --out results/preds_vllm.jsonl

In [ ]:
!zip -r results.zip results
from google.colab import files
files.download("results.zip")